# What is the dependency on the galaxy selection?
--------

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from get_model_probabilities import *
import scienceplots
plt.style.use(["science","grid"])
from sidm_inference_on_data_and_models import infer_sidm,infer_sidm_interp
from scipy.stats import binned_statistic
from add_shear_to_data import combine_catalogues, get_obs_data, get_model_names

DEFAULT ZS 2.28808070104492
Source redshift:2.2884950162268827


In [4]:
ifilter = 'concat'
all_mag_low_cuts = np.linspace(21,24,11)
all_mag_cuts = np.linspace(28,26,11)
all_siz_cuts = np.linspace(2,3,11)

# First look at the change in the observed value as you change magnitude and size cuts

In [10]:

outfile = f"../data/100/a2744/obs_data_{ifilter}_gal_selection_bias.pkl"
obs_data = get_obs_data( ifilter, data_dir="../data/100/a2744/")

binned_data = bin_obs_data( obs_data )

fid_e1 = binned_data['e1']
fid_e2 = binned_data['e2']

e1_stacked = [fid_e1]
e2_stacked = [fid_e2]
ngalaxies = []
for iMAG_CUT in all_mag_low_cuts:
    
    new_data = obs_data.copy()    

    
    new_data = new_data[ new_data['MAG'] > iMAG_CUT]
    
    binned_data = bin_obs_data( new_data )
   
    cut_e1 = binned_data['e1']
    cut_e2 = binned_data['e2']

    e1_stacked.append( cut_e1 )
    e2_stacked.append( cut_e2 )
    ngalaxies.append(new_data.shape[0])
    
for iMAG_CUT in all_mag_cuts:
    
    new_data = obs_data.copy()    

    
    new_data = new_data[ new_data['MAG'] < iMAG_CUT]
    
    binned_data = bin_obs_data( new_data )
   
    cut_e1 = binned_data['e1']
    cut_e2 = binned_data['e2']
    
    e1_stacked.append( cut_e1 )
    e2_stacked.append( cut_e2 )
    ngalaxies.append(new_data.shape[0])
for iSIZE_CUT in all_siz_cuts:
    
    new_data = obs_data.copy()
    
    new_data = new_data[ new_data['SIZE'] > iSIZE_CUT]
    

    binned_data = bin_obs_data( new_data )
   
    cut_e1 = binned_data['e1']
    cut_e2 = binned_data['e2']
    
    e1_stacked.append( cut_e1 )
    e2_stacked.append( cut_e2 )
    ngalaxies.append(new_data.shape[0])
 
e1_stacked = np.array(e1_stacked)
e2_stacked = np.array(e2_stacked)

stacked = np.append(e1_stacked[None,:,:,:], e2_stacked[None,:,:,:], axis=0)
stacked = np.moveaxis( stacked, 0, 1)
ngalaxies = np.array(ngalaxies)

pkl.dump([{}, stacked],open(outfile,"wb"))

#RUN ALL MODELS ON THE TEST DATA
#-------------------------------
args.source_domain='a2744'
args.target_domain='a2744'

models = {}
probabilities = {}
probabilities_noise = {}

    
    
all_models, seed_index =  get_model_names(model='tweakedalign_best', 
                                              load_model_args=args,
                                             )
models[ifilter] = all_models
meta, data = pkl.load(open(outfile,"rb"))

#repeat 
probabilities_filt = []
probabilities_noise_filt = []

with torch.no_grad():

    for imodel in tqdm(all_models):
        outputs_dict = imodel([torch.tensor(data,dtype=torch.float32)])
        #if torch.all(torch.abs(outputs_dict['classification'][:,0] - outputs_dict['classification'][0,0])  < 0.01):
        #   continue
        probabilities_filt.append(torch.softmax( outputs_dict['classification'], dim=1 )[0,0]) 
        probabilities_noise_filt.append(torch.softmax( outputs_dict['classification'], dim=1 )[1:,0])


probabilities_filt = torch.tensor( probabilities_filt).detach().numpy()
probabilities_noise_filt =  torch.stack( probabilities_noise_filt).detach().numpy()


probabilities[ifilter] = probabilities_filt
probabilities_noise[ifilter] = probabilities_noise_filt
pkl.dump([models, probabilities, probabilities_noise, ngalaxies], open("pickles/model_on_data_galaxy_selection.pkl","wb"))

100%|███████████████████████████████████████████████████████████████████████████████████| 30/30 [00:07<00:00,  4.25it/s]


# Now find the new thresholds for blocking out different amounts of galaxies

In [ ]:
imonte=5
all_results = {}

ngal_list = ngalaxies[11:]/ngalaxies[0]

for ifilter in ['concat']:
    already_done = 0
    output_name = f"pickles/all_models_{ifilter}_ngal_dep_results.pkl"
    
    fid_n_z = get_variable_redshift_dist(data_dir="../data/100/a2744/", ifilter='concat')
        
    if os.path.isfile(output_name):
        all_results = pkl.load(open(output_name,'rb'))
    else:
        all_results = {}
    all_results = {}
           
    for iNgal in ngal_list[::-1]:
        
        if iNgal == 1:
            if already_done ==0: 
                already_done = 1
            else:
                continue
        
        all_results[f"{iNgal:0.2f}"] = {}
        
        image_size=100

        n_z = fid_n_z.copy()

        # indices where arr != 0
        valid_idx = np.argwhere( n_z != np.median(n_z))
        
        
        number_to_block = int(len(valid_idx)*(1-iNgal)) 

        chosen = valid_idx[
            np.random.choice(
                len(valid_idx),
                size=number_to_block,
                replace=False
            )
        ]

       
        
        n_z[chosen[:,0], chosen[:,1]] = 0
        

        all_models, seed_index =  get_model_names(model='tweakedalign_best', nmodels=5)

        for imodel in tqdm(all_models):

            seed = imodel.split('_')[seed_index]

            args.jwst_filter = ifilter
            args.apply_intrinsic_ell = 1.


            domain = {
            'tgt':'darkskies_obs',
            'src':'bahamas_obs'
            }
            args.ignore_dataset = [''] # Although i ignored during training i want to see duringn testing.

            if f"seed_{seed}" not in all_results[f"{iNgal:0.2f}"].keys():

                all_results[f"{iNgal:0.2f}"][f"seed_{seed}"]  = {'src':[],'tgt':[]}

            args.unbalance = True

            for i in range(imonte):
                args.zs = n_z

                for idomain in domain.keys():

                    target_domain = domain[idomain]
                    results = get_probabilities( 
                            target_domain,
                            [imodel],
                            args,
                            quiet=True
                    )
                    del results['data_loaders']

                    all_results[f"{iNgal:0.2f}"][f"seed_{seed}"][idomain].append( results )

            pkl.dump(all_results, open(output_name,"wb"))


# Cluster Member Impact
-----------
Get the new thresholds in the case where we add galaxy members to the lensing signal

In [ ]:

ifilter = 'concat'

all_models, seed_index =  get_model_names(model='tweakedalign_best', nmodels=5)
imonte=5


output_name = f"pickles/cluster_contamination_{ifilter}_test.pkl"

if os.path.isfile( output_name ):
    all_results = pkl.load(open(output_name,"rb"))
else:
    all_results = {}


domain = {
    'tgt':'darkskies_obs',
    'src':'bahamas_obs'
}
args.jwst_filter = 'concat'
args.apply_intrinsic_ell = 1.


args.ignore_dataset = [''] # Althopiugh i ignored during training i want to see duringn testing.

args.unbalance = True
contaminant_list = np.linspace(0,0.2,6)
for cluster_contamination in contaminant_list:
    
    if cluster_contamination not in all_results.keys():
        all_results[cluster_contamination] = {}
    
    args.cluster_member_contamination = cluster_contamination
    
    for imodel in tqdm(all_models):

        seed = imodel.split('_')[seed_index]

        if f"seed_{seed}" not in all_results[cluster_contamination].keys():
            all_results[cluster_contamination][f"seed_{seed}"]  = {'src':[],'tgt':[]}
        


        ndone = len(all_results[cluster_contamination][f"seed_{seed}"]['src'])
        
        for i in range(ndone,imonte):
            for idomain in domain.keys():

                target_domain = domain[idomain]
                results = get_probabilities( 
                        target_domain,
                        [imodel],
                        args,
                        quiet=True
                )
                del results['data_loaders']

                all_results[cluster_contamination][f"seed_{seed}"][idomain].append( results )

    pkl.dump(all_results, open(output_name,"wb"))
